# Get More Data

Pull recent IBM and IonQ job history, save a unified table, and build a merge-ready runtime CSV for the rest of the pipeline.

Outputs:
- `final_data/historic_runs_unified.csv`
- `final_data/historic_runs_unified.json`
- `final_data/historic_runs_for_estimate_runtime_df.csv`

In [9]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
import yaml
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

BASE = Path('/lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/MQBac')
OUT_DIR = BASE / 'final_data'
OUT_DIR.mkdir(parents=True, exist_ok=True)

IBM_YAML = Path('/lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/QFw/DEFw/python/services/svc_ibmq_qpm/ibmq_env.yaml')
IONQ_YAML = Path('/lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/QFw/DEFw/python/services/svc_ionq_qpm/ionq_env.yaml')


def load_cfg(path: Path) -> dict:
    data = yaml.safe_load(path.read_text()) or {}
    return data.get('CONFIG', {}) if isinstance(data, dict) else {}


ibm_cfg = load_cfg(IBM_YAML)
ionq_cfg = load_cfg(IONQ_YAML)

IBMQ_API_KEY = (ibm_cfg.get('IBMQ_API_KEY') or os.environ.get('IBMQ_API_KEY') or os.environ.get('QISKIT_IBM_TOKEN') or '').strip()
IBMQ_INSTANCE = (ibm_cfg.get('IBMQ_SERVICE_CRN') or os.environ.get('IBMQ_SERVICE_CRN') or os.environ.get('QISKIT_IBM_INSTANCE') or '').strip()
IBMQ_CHANNEL = (ibm_cfg.get('IBMQ_CHANNEL') or os.environ.get('IBMQ_CHANNEL') or 'ibm_quantum_platform').strip()
IONQ_API_KEY = (ionq_cfg.get('IONQ_API_KEY') or os.environ.get('IONQ_API_KEY') or '').strip()
HTTPS_PROXY = (ibm_cfg.get('HTTPS_PROXY') or ionq_cfg.get('HTTPS_PROXY') or os.environ.get('HTTPS_PROXY') or '').strip()
HTTP_PROXY = (ionq_cfg.get('HTTP_PROXY') or os.environ.get('HTTP_PROXY') or HTTPS_PROXY or '').strip()

proxies = {}
if HTTP_PROXY:
    proxies['http'] = HTTP_PROXY
if HTTPS_PROXY:
    proxies['https'] = HTTPS_PROXY

os.environ['QISKIT_IBM_TOKEN'] = IBMQ_API_KEY
os.environ['QISKIT_IBM_INSTANCE'] = IBMQ_INSTANCE
os.environ['QISKIT_IBM_CHANNEL'] = IBMQ_CHANNEL
if HTTP_PROXY:
    os.environ['HTTP_PROXY'] = HTTP_PROXY
if HTTPS_PROXY:
    os.environ['HTTPS_PROXY'] = HTTPS_PROXY


def pick(d, *keys):
    for key in keys:
        if isinstance(d, dict) and key in d and d[key] is not None:
            return d[key]
    return None


def deep_pick(obj, keys_set):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in keys_set and v is not None:
                return v
            out = deep_pick(v, keys_set)
            if out is not None:
                return out
    elif isinstance(obj, list):
        for item in obj:
            out = deep_pick(item, keys_set)
            if out is not None:
                return out
    return None


def deep_path_pick(obj, path):
    cur = obj
    for p in path:
        if isinstance(cur, dict):
            if p not in cur:
                return None
            cur = cur[p]
        elif isinstance(cur, list):
            if not isinstance(p, int) or p < 0 or p >= len(cur):
                return None
            cur = cur[p]
        else:
            return None
    return cur


def to_dt(value):
    if value is None or value == '':
        return None
    if isinstance(value, (int, float)):
        if value > 10_000_000_000:
            value = value / 1000.0
        return datetime.fromtimestamp(value, tz=timezone.utc)
    text = str(value).strip().replace('Z', '+00:00')
    try:
        dt = datetime.fromisoformat(text)
        return dt if dt.tzinfo else dt.replace(tzinfo=timezone.utc)
    except ValueError:
        return None


def runtime_ms(start, end):
    start_dt = to_dt(start)
    end_dt = to_dt(end)
    if start_dt is None or end_dt is None:
        return None
    return (end_dt - start_dt).total_seconds() * 1000.0


def walk(obj):
    if isinstance(obj, dict):
        for value in obj.values():
            yield from walk(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk(value)
    else:
        yield obj


def find_qasm(raw):
    for key in ['qasm', 'openqasm', 'qasm_str', 'qasm_string']:
        value = raw.get(key)
        if isinstance(value, str) and ('OPENQASM' in value or 'qreg' in value):
            return value
    for value in walk(raw):
        if isinstance(value, str) and ('OPENQASM' in value or ('qreg' in value and ';' in value)):
            return value
    return None


def find_instructions(raw):
    if isinstance(raw, dict):
        if isinstance(raw.get('instructions'), list):
            return raw['instructions']
        for value in raw.values():
            found = find_instructions(value)
            if found is not None:
                return found
    elif isinstance(raw, list):
        for value in raw:
            found = find_instructions(value)
            if found is not None:
                return found
    return None


def maybe_load_qasm_string(qasm_text):
    if not isinstance(qasm_text, str) or not qasm_text.strip():
        return None
    text = qasm_text.strip()
    try:
        if 'OPENQASM 3' in text.upper():
            from qiskit.qasm3 import loads as loads_qasm3
            return loads_qasm3(text)
        return QuantumCircuit.from_qasm_str(text)
    except Exception:
        return None


def find_qiskit_circuit(raw):
    qasm_text = find_qasm(raw)
    if qasm_text:
        qc = maybe_load_qasm_string(qasm_text)
        if qc is not None:
            return qc

    pub_circ = (
        deep_path_pick(raw, ['params', 'pubs', 0, 'circuit'])
        or deep_path_pick(raw, ['inputs', 'pubs', 0, 'circuit'])
        or deep_path_pick(raw, ['job_detail', 'params', 'pubs', 0, 'circuit'])
    )
    if isinstance(pub_circ, str):
        qc = maybe_load_qasm_string(pub_circ)
        if qc is not None:
            return qc
    if isinstance(pub_circ, dict):
        for k in ['qasm', 'openqasm', 'qasm_str', 'qasm_string']:
            qc = maybe_load_qasm_string(pub_circ.get(k))
            if qc is not None:
                return qc

    return None


def qiskit_detail_metrics(qc):
    if qc is None:
        return {}

    clifford_ops = {
        'id', 'x', 'y', 'z', 'h', 's', 'sdg', 'sx', 'sxdg',
        'cx', 'cy', 'cz', 'swap', 'measure', 'barrier', 'reset'
    }

    op_names = []
    used_qubits = set()
    edges = set()
    for inst, qargs, _ in qc.data:
        name = str(inst.name).lower()
        op_names.append(name)
        idxs = []
        for q in qargs:
            qi = qc.find_bit(q).index
            idxs.append(qi)
            used_qubits.add(qi)
        if len(idxs) >= 2:
            for i in range(len(idxs)):
                for j in range(i + 1, len(idxs)):
                    a, b = sorted((idxs[i], idxs[j]))
                    edges.add((a, b))

    n_cliff = sum(1 for n in op_names if n in clifford_ops)
    n_non = len(op_names) - n_cliff

    # Connected components in the interaction graph of used qubits.
    if not used_qubits:
        n_comp = 0
    else:
        parent = {u: u for u in used_qubits}

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra

        for a, b in edges:
            if a in parent and b in parent:
                union(a, b)

        n_comp = len({find(u) for u in used_qubits})

    return {
        'framework': 'qiskit',
        'critical_path_length': int(qc.depth()),
        'connected_components': int(n_comp),
        'num_cliffords': int(n_cliff),
        'num_non_cliffords': int(n_non),
        'num_parameters': int(len(qc.parameters)),
    }


def extract_features(raw, shots=None):
    out = {
        'benchmark_hint': None,
        'input_format': None,
        'n_qubits': pick(raw, 'qubits', 'n_qubits', 'num_qubits') or deep_pick(raw, {'qubits', 'n_qubits', 'num_qubits'}),
        'depth': None,
        'n_ops': None,
        'single_qubit_gates': None,
        'two_qubit_gates': None,
        'three_qubit_gates': None,
        'measure_ops': None,
        'gate_counts_json': None,
        'critical_path_length': None,
        'connected_components': None,
        'num_cliffords': None,
        'num_non_cliffords': None,
        'num_parameters': None,
        'framework': None,
    }

    text = ' '.join([
        str(pick(raw, 'name') or ''),
        str(pick(raw, 'job_name') or ''),
        str(pick(raw, 'description') or ''),
        str(pick(raw, 'backend') or ''),
        str(pick(raw, 'target') or ''),
    ]).lower()
    for name in ['ghz', 'ham', 'mermin_bell', 'bit_code', 'phase_code', 'qaoa', 'vqe', 'hhl', 'tfim']:
        if name in text:
            out['benchmark_hint'] = name
            break

    instructions = find_instructions(raw)
    if instructions:
        gate_counts = {}
        one_q = 0
        two_q = 0
        three_q = 0
        meas = 0
        max_idx = -1
        for ins in instructions:
            if not isinstance(ins, dict):
                continue
            op = str(ins.get('gate') or ins.get('name') or ins.get('op') or 'unknown')
            qubits = ins.get('target') or ins.get('targets') or ins.get('qubits') or []
            if isinstance(qubits, int):
                qubits = [qubits]
            if not isinstance(qubits, list):
                qubits = []
            for qubit in qubits:
                try:
                    max_idx = max(max_idx, int(qubit))
                except Exception:
                    pass
            gate_counts[op] = gate_counts.get(op, 0) + 1
            if op.lower() in {'measure', 'measurement', 'mz', 'readout'}:
                meas += 1
            if len(qubits) == 1:
                one_q += 1
            elif len(qubits) == 2:
                two_q += 1
            elif len(qubits) >= 3:
                three_q += 1
        out['input_format'] = 'instructions'
        out['n_ops'] = sum(gate_counts.values()) or None
        out['single_qubit_gates'] = one_q
        out['two_qubit_gates'] = two_q
        out['three_qubit_gates'] = three_q
        out['measure_ops'] = meas
        out['gate_counts_json'] = json.dumps(gate_counts) if gate_counts else None
        if out['n_qubits'] is None and max_idx >= 0:
            out['n_qubits'] = max_idx + 1

    qc = find_qiskit_circuit(raw)
    if qc is not None:
        counts = qc.count_ops()
        one_q = 0
        two_q = 0
        three_q = 0
        for inst, qargs, _ in qc.data:
            if len(qargs) == 1:
                one_q += 1
            elif len(qargs) == 2:
                two_q += 1
            elif len(qargs) >= 3:
                three_q += 1

        out['input_format'] = 'qasm'
        out['n_qubits'] = int(qc.num_qubits)
        out['depth'] = int(qc.depth())
        out['n_ops'] = int(qc.size())
        out['single_qubit_gates'] = one_q
        out['two_qubit_gates'] = two_q
        out['three_qubit_gates'] = three_q
        out['measure_ops'] = int(counts.get('measure', 0))
        out['gate_counts_json'] = json.dumps({str(k): int(v) for k, v in counts.items()})

        qm = qiskit_detail_metrics(qc)
        if qm:
            out['framework'] = qm.get('framework')
            out['critical_path_length'] = qm.get('critical_path_length')
            out['connected_components'] = qm.get('connected_components')
            out['num_cliffords'] = qm.get('num_cliffords')
            out['num_non_cliffords'] = qm.get('num_non_cliffords')
            out['num_parameters'] = qm.get('num_parameters')
            out['circuit_metrics_json'] = json.dumps(qm)

    return out


print('Output dir:', OUT_DIR)
print('IBM channel:', IBMQ_CHANNEL)
print('Proxies:', list(proxies.keys()))
print('Qiskit detail metrics enabled:', True)

Output dir: /lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/MQBac/final_data
IBM channel: ibm_quantum_platform
Proxies: ['http', 'https']
Qiskit detail metrics enabled: True


In [10]:
from qiskit import transpile


def _coerce_qc(obj):
    if isinstance(obj, QuantumCircuit):
        return obj
    if isinstance(obj, str):
        return maybe_load_qasm_string(obj)
    if isinstance(obj, dict):
        for k in ['qasm', 'openqasm', 'qasm_str', 'qasm_string']:
            qc = maybe_load_qasm_string(obj.get(k))
            if qc is not None:
                return qc
    return None


def _runtime_job_input_circuits(job):
    try:
        inputs = job.inputs
    except Exception:
        return []

    cands = []
    if isinstance(inputs, dict):
        circs = inputs.get('circuits')
        if isinstance(circs, list):
            cands.extend(circs)
        elif circs is not None:
            cands.append(circs)

        pubs = inputs.get('pubs')
        if isinstance(pubs, list):
            for pub in pubs:
                if isinstance(pub, dict) and 'circuit' in pub:
                    cands.append(pub['circuit'])
                elif isinstance(pub, (list, tuple)) and len(pub) > 0:
                    cands.append(pub[0])

    out = []
    for c in cands:
        qc = _coerce_qc(c)
        if qc is not None:
            out.append(qc)
    return out


def _basic_qc_metrics(qc):
    counts = qc.count_ops()
    one_q = 0
    two_q = 0
    three_q = 0
    for inst, qargs, _ in qc.data:
        if len(qargs) == 1:
            one_q += 1
        elif len(qargs) == 2:
            two_q += 1
        elif len(qargs) >= 3:
            three_q += 1
    return {
        'input_format': 'runtime_inputs',
        'n_qubits': int(qc.num_qubits),
        'depth': int(qc.depth()),
        'n_ops': int(qc.size()),
        'single_qubit_gates': one_q,
        'two_qubit_gates': two_q,
        'three_qubit_gates': three_q,
        'measure_ops': int(counts.get('measure', 0)),
        'gate_counts_json': json.dumps({str(k): int(v) for k, v in counts.items()}),
    }


def runtime_job_circuit_features(service, job_id, backend_name=None, shots=None):
    if not job_id:
        return {}
    try:
        job = service.job(job_id)
        circuits = _runtime_job_input_circuits(job)
        if not circuits:
            return {}

        qc = circuits[0]
        fmt = 'runtime_inputs'
        if backend_name:
            try:
                backend_obj = service.backend(backend_name)
                qc = transpile(qc, backend=backend_obj, optimization_level=1)
                fmt = 'transpiled'
            except Exception:
                pass

        out = _basic_qc_metrics(qc)
        out['input_format'] = fmt

        qm = qiskit_detail_metrics(qc)
        if qm:
            out['circuit_metrics_json'] = json.dumps(qm)
            out['framework'] = qm.get('framework')
            out['critical_path_length'] = qm.get('critical_path_length')
            out['connected_components'] = qm.get('connected_components')
            out['num_cliffords'] = qm.get('num_cliffords')
            out['num_non_cliffords'] = qm.get('num_non_cliffords')
            out['num_parameters'] = qm.get('num_parameters')
        return out
    except Exception:
        return {}

In [11]:
import time

service = QiskitRuntimeService(
    channel=IBMQ_CHANNEL,
    token=IBMQ_API_KEY,
    instance=IBMQ_INSTANCE,
)

# 1) Pull recent job summaries (small pages for reliability)
ibm_summaries = []
limit = 80
skip = 0
page_size = 20
max_retries = 5

while len(ibm_summaries) < limit:
    batch_size = min(page_size, limit - len(ibm_summaries))

    ok = False
    for attempt in range(max_retries):
        try:
            resp = service._active_api_client.jobs_get(limit=batch_size, skip=skip, descending=True)
            ok = True
            break
        except Exception as e:
            msg = str(e)
            if '503' in msg or 'Service Unavailable' in msg:
                wait_s = 2 ** attempt
                print(f'IBM jobs_get 503, retrying in {wait_s}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait_s)
                continue
            raise

    if not ok:
        print('Stopping IBM summary fetch after repeated 503 responses.')
        break

    jobs = resp.get('jobs', [])
    if not jobs:
        break
    ibm_summaries.extend(jobs)
    skip += len(jobs)

# 2) Enrich each job with full detail + metadata so runtime/circuit fields are available
ibm_raw = []
for summary in ibm_summaries:
    job_id = pick(summary, 'id', 'job_id')
    if not job_id:
        continue

    detail = {}
    meta = {}

    for attempt in range(max_retries):
        try:
            detail = service._active_api_client.job_get(job_id=job_id, exclude_params=False) or {}
            break
        except Exception as e:
            msg = str(e)
            if '503' in msg or 'Service Unavailable' in msg:
                time.sleep(2 ** attempt)
                continue
            break

    for attempt in range(max_retries):
        try:
            meta = service._active_api_client.job_metadata(job_id=job_id) or {}
            break
        except Exception as e:
            msg = str(e)
            if '503' in msg or 'Service Unavailable' in msg:
                time.sleep(2 ** attempt)
                continue
            break

    merged = dict(summary)
    merged['job_detail'] = detail
    merged['job_metadata'] = meta

    # Flatten top-level detail keys for simpler extraction later
    for k, v in detail.items():
        if k not in merged or merged[k] in (None, '', {}):
            merged[k] = v

    ibm_raw.append(merged)

# 3) Normalize rows for downstream 0_preprocess merge schema
ibm_rows = []
for raw in ibm_raw:
    start = (
        pick(raw, 'started', 'started_at', 'running_at')
        or deep_path_pick(raw, ['state', 'timestamps', 'running'])
        or deep_path_pick(raw, ['job_metadata', 'timestamps', 'running'])
        or deep_pick(raw, {'running', 'running_at', 'started_at', 'start_time'})
    )
    end = (
        pick(raw, 'completed', 'completed_at', 'ended_at', 'finished_at')
        or deep_path_pick(raw, ['state', 'timestamps', 'finished'])
        or deep_path_pick(raw, ['job_metadata', 'timestamps', 'finished'])
        or deep_pick(raw, {'finished', 'finished_at', 'ended_at', 'completed_at'})
    )

    val = runtime_ms(start, end)
    if val is None:
        sec = (
            pick(raw, 'estimated_running_time_seconds')
            or deep_path_pick(raw, ['job_detail', 'estimated_running_time_seconds'])
            or deep_path_pick(raw, ['job_metadata', 'usage', 'quantum_seconds'])
            or deep_pick(raw, {'estimated_running_time_seconds', 'quantum_seconds'})
        )
        if isinstance(sec, (int, float)):
            val = float(sec) * 1000.0

    shots = (
        pick(raw, 'shots')
        or deep_path_pick(raw, ['params', 'shots'])
        or deep_path_pick(raw, ['params', 'pubs', 0, 'shots'])
        or deep_path_pick(raw, ['inputs', 'run_options', 'shots'])
        or deep_pick(raw, {'shots', 'num_shots'})
    )

    backend_name = pick(raw, 'backend') or deep_pick(raw, {'backend', 'backend_name', 'target'})
    feats = extract_features(raw, shots=shots)

    # Minimal fallback: use RuntimeJob.inputs['circuits'] and transpile to backend for richer Qiskit metrics.
    if feats.get('n_qubits') is None or feats.get('critical_path_length') is None:
        extra = runtime_job_circuit_features(
            service=service,
            job_id=pick(raw, 'id', 'job_id'),
            backend_name=backend_name,
            shots=shots,
        )
        if extra:
            for k, v in extra.items():
                if feats.get(k) in (None, '', {}):
                    feats[k] = v

    ibm_rows.append({
        'provider': 'ibmq',
        'job_id': pick(raw, 'id', 'job_id'),
        'backend': backend_name,
        'status': pick(raw, 'status', 'state') or deep_path_pick(raw, ['state', 'status']) or deep_pick(raw, {'status', 'state'}),
        'created_at': pick(raw, 'created', 'created_at') or deep_pick(raw, {'created', 'created_at', 'creation_date'}),
        'started_at': start,
        'ended_at': end,
        'runtime_ms': val,
        'shots': shots,
        'benchmark_hint': feats['benchmark_hint'],
        'input_format': feats['input_format'],
        'n_qubits': feats['n_qubits'],
        'depth': feats['depth'],
        'n_ops': feats['n_ops'],
        'single_qubit_gates': feats['single_qubit_gates'],
        'two_qubit_gates': feats['two_qubit_gates'],
        'three_qubit_gates': feats['three_qubit_gates'],
        'measure_ops': feats['measure_ops'],
        'gate_counts_json': feats['gate_counts_json'],
        'qbacmet_framework': feats['framework'],
        'critical_path_length': feats['critical_path_length'],
        'connected_components': feats['connected_components'],
        'num_cliffords': feats['num_cliffords'],
        'num_non_cliffords': feats['num_non_cliffords'],
        'num_parameters': feats['num_parameters'],
        'circuit_metrics_json': feats.get('circuit_metrics_json'),
        'session_id': pick(raw, 'session_id') or deep_pick(raw, {'session_id'}),
    })

ibm_df = pd.DataFrame(ibm_rows)
print('IBM summaries fetched:', len(ibm_summaries))
print('IBM detailed rows:', len(ibm_df))
print('IBM rows with runtime_ms:', int(ibm_df['runtime_ms'].notna().sum()) if 'runtime_ms' in ibm_df else 0)
print('IBM rows with circuit features (n_qubits):', int(ibm_df['n_qubits'].notna().sum()) if 'n_qubits' in ibm_df else 0)
print('IBM rows with Qiskit details:', int(ibm_df['critical_path_length'].notna().sum()) if 'critical_path_length' in ibm_df else 0)
print('IBM rows with full circuit metrics json:', int(ibm_df['circuit_metrics_json'].notna().sum()) if 'circuit_metrics_json' in ibm_df else 0)
ibm_df.head()

qiskit_runtime_service._discover_account:WARNING:2026-04-21 02:57:58,074: Loading account with the given token. A saved account will not be used.
/tmp/ipykernel_3213105/603533533.py:52: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for inst, qargs, _ in qc.data:
/tmp/ipykernel_3213105/2545336095.py:207: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for inst, qargs, _ in qc.data:
qiskit_runtime_service._create_backend_obj:WARNING:2026-04-21 02:58:57,430: Unable to create configuration for ibm_torino. '404 Client Error: Not Found for url: https://quantum.cloud.ibm.com/api/v1/backends/ibm_torino/configuration. {"errors":[{"code":"not_found","message":

IBM summaries fetched: 80
IBM detailed rows: 80
IBM rows with runtime_ms: 80
IBM rows with circuit features (n_qubits): 80
IBM rows with Qiskit details: 80
IBM rows with full circuit metrics json: 80


,provider,job_id,backend,status,created_at,started_at,ended_at,runtime_ms,shots,benchmark_hint,...,measure_ops,gate_counts_json,qbacmet_framework,critical_path_length,connected_components,num_cliffords,num_non_cliffords,num_parameters,circuit_metrics_json,session_id
0,ibmq,d6gvatithhns7391g1eg,ibm_pittsburgh,Completed,2026-02-27T19:50:14.591476Z,2026-02-27T19:50:15.97821Z,2026-02-27T19:50:50.220745Z,34242.535,None,None,...,24,"{""sx"": 3146, ""rz"": 2672, ""cz"": 1089, ""x"": 121,...",qiskit,512,1,4381,2672,0,"{""framework"": ""qiskit"", ""critical_path_length""...",NaN
1,ibmq,d6gv5k1keb2s73bdvlm0,ibm_pittsburgh,Completed,2026-02-27T19:38:56.596788Z,2026-02-27T19:38:57.672018Z,2026-02-27T19:39:31.487533Z,33815.515,None,None,...,24,"{""sx"": 2200, ""rz"": 1759, ""cz"": 730, ""x"": 25, ""...",qiskit,412,1,2980,1759,0,"{""framework"": ""qiskit"", ""critical_path_length""...",NaN
2,ibmq,d6gv4m9keb2s73bdvkkg,ibm_pittsburgh,Completed,2026-02-27T19:36:58.134581Z,2026-02-27T19:36:59.060995Z,2026-02-27T19:37:33.566454Z,34505.459,None,None,...,24,"{""sx"": 2209, ""rz"": 1738, ""cz"": 744, ""x"": 32, ""...",qiskit,445,1,3010,1738,0,"{""framework"": ""qiskit"", ""critical_path_length""...",NaN
3,ibmq,d6gv1ae48nic73amc8b0,ibm_pittsburgh,Completed,2026-02-27T19:29:45.311504Z,2026-02-27T19:29:46.057064Z,2026-02-27T19:30:19.79415Z,33737.086,None,None,...,24,"{""sx"": 2145, ""rz"": 1754, ""cz"": 700, ""measure"":...",qiskit,383,1,2892,1754,0,"{""framework"": ""qiskit"", ""critical_path_length""...",NaN
4,ibmq,d6guuvn3o3rs73cabc7g,ibm_pittsburgh,Completed,2026-02-27T19:24:46.503387Z,2026-02-27T19:24:47.381003Z,2026-02-27T19:25:21.293062Z,33912.059,None,None,...,24,"{""sx"": 1354, ""rz"": 1088, ""cz"": 436, ""measure"":...",qiskit,245,1,1832,1088,0,"{""framework"": ""qiskit"", ""critical_path_length""...",NaN


In [12]:
ionq_rows = []

try:
    from qiskit_ionq import IonQProvider

    ionq_provider = IonQProvider(IONQ_API_KEY)
    ionq_backends = ionq_provider.backends()
    print('IonQ provider backends:', [b.name() if callable(getattr(b, 'name', None)) else str(getattr(b, 'name', 'unknown')) for b in ionq_backends])

    ionq_jobs = []
    seen_ids = set()

    for backend_obj in ionq_backends:
        try:
            try:
                jobs = backend_obj.jobs(limit=40)
            except TypeError:
                jobs = backend_obj.jobs()
        except Exception:
            continue

        for job in jobs or []:
            try:
                jid = job.job_id() if hasattr(job, 'job_id') else None
            except Exception:
                jid = None
            if not jid or jid in seen_ids:
                continue
            seen_ids.add(jid)
            ionq_jobs.append((backend_obj, job))
            if len(ionq_jobs) >= 80:
                break
        if len(ionq_jobs) >= 80:
            break

    for backend_obj, job in ionq_jobs:
        raw = {}

        # Best effort: pull any raw payload exposed by provider internals.
        try:
            maybe_raw = getattr(job, '_job', None)
            if isinstance(maybe_raw, dict):
                raw.update(maybe_raw)
        except Exception:
            pass

        try:
            meta = job.metadata() if hasattr(job, 'metadata') else {}
            if isinstance(meta, dict):
                for k, v in meta.items():
                    if k not in raw:
                        raw[k] = v
        except Exception:
            pass

        try:
            backend_name = backend_obj.name() if callable(getattr(backend_obj, 'name', None)) else str(getattr(backend_obj, 'name', 'ionq'))
        except Exception:
            backend_name = 'ionq'

        try:
            job_id = job.job_id() if hasattr(job, 'job_id') else pick(raw, 'id', 'job_id')
        except Exception:
            job_id = pick(raw, 'id', 'job_id')

        try:
            status = str(job.status())
        except Exception:
            status = pick(raw, 'status', 'state') or deep_pick(raw, {'status', 'state'})

        created_at = (
            str(getattr(job, 'creation_date')) if getattr(job, 'creation_date', None) is not None
            else pick(raw, 'created', 'created_at') or deep_pick(raw, {'created', 'created_at', 'creation_time'})
        )
        started_at = pick(raw, 'started', 'started_at') or deep_pick(raw, {'started', 'started_at', 'running_at', 'start_time'})
        ended_at = pick(raw, 'completed', 'completed_at', 'ended_at') or deep_pick(raw, {'completed', 'completed_at', 'ended_at', 'finish_time'})

        value = runtime_ms(started_at, ended_at)
        if value is None:
            value = (
                pick(raw, 'execution_time', 'execution_time_ms', 'runtime_ms')
                or deep_path_pick(raw, ['execution', 'duration'])
                or deep_pick(raw, {'execution_time', 'execution_time_ms', 'runtime_ms', 'duration'})
            )
            if isinstance(value, (int, float)) and value < 10000:
                value = value * 1000.0

        shots = (
            pick(raw, 'shots')
            or deep_path_pick(raw, ['metadata', 'shots'])
            or deep_path_pick(raw, ['input', 'shots'])
            or deep_path_pick(raw, ['input', 'params', 'shots'])
            or deep_pick(raw, {'shots', 'num_shots'})
        )

        feats = extract_features(raw, shots=shots)
        ionq_rows.append({
            'provider': 'ionq',
            'job_id': job_id,
            'backend': backend_name,
            'status': status,
            'created_at': created_at,
            'started_at': started_at,
            'ended_at': ended_at,
            'runtime_ms': value,
            'shots': shots,
            'benchmark_hint': feats['benchmark_hint'],
            'input_format': feats['input_format'],
            'n_qubits': feats['n_qubits'],
            'depth': feats['depth'],
            'n_ops': feats['n_ops'],
            'single_qubit_gates': feats['single_qubit_gates'],
            'two_qubit_gates': feats['two_qubit_gates'],
            'three_qubit_gates': feats['three_qubit_gates'],
            'measure_ops': feats['measure_ops'],
            'gate_counts_json': feats['gate_counts_json'],
            'session_id': None,
        })

except Exception as e:
    print('IonQ provider-based fetch failed:', e)

ionq_df = pd.DataFrame(ionq_rows)
print('IonQ jobs fetched:', len(ionq_df))
print('IonQ rows with shots:', int(ionq_df['shots'].notna().sum()) if 'shots' in ionq_df else 0)
print('IonQ rows with runtime_ms:', int(ionq_df['runtime_ms'].notna().sum()) if 'runtime_ms' in ionq_df else 0)
ionq_df.head()

/lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/MQBac/mqbacVirtEnv/lib64/python3.11/site-packages/qiskit_ionq/ionq_backend.py:127: IonQTranspileLevelWarning: Transpiler default optimization_level=2. IonQ (QIS) recommends 0-1 to avoid aggressive re-synthesis; use transpile(..., optimization_level=1).
  warn_bad_transpile_level()


IonQ provider backends: ['ionq_simulator', 'ionq_qpu']
IonQ jobs fetched: 0
IonQ rows with shots: 0
IonQ rows with runtime_ms: 0


""


In [13]:
from qiskit_ionq import IonQProvider
import inspect
import json as _json
from pprint import pprint

provider = IonQProvider(IONQ_API_KEY)
backs = provider.backends()
print('backends', [b.name() if callable(getattr(b, 'name', None)) else getattr(b, 'name', str(b)) for b in backs])

sample = None
sample_backend = None
for b in backs:
    try:
        jobs = b.jobs(limit=3)
    except TypeError:
        jobs = b.jobs()
    except Exception as e:
        print('jobs failed for backend', b, e)
        continue
    jobs = list(jobs or [])
    print('backend jobs', b, len(jobs))
    if jobs:
        sample = jobs[0]
        sample_backend = b
        break

print('sample_backend', sample_backend)
print('sample_type', type(sample))
print('sample_dir', [x for x in dir(sample) if not x.startswith('__')][:200])

if sample is not None:
    for name in ['job_id', 'status', 'creation_date', 'metadata', 'result', 'backend', 'backend_options']:
        try:
            attr = getattr(sample, name)
            if callable(attr):
                value = attr()
            else:
                value = attr
            print('\nFIELD', name, 'TYPE', type(value))
            if isinstance(value, dict):
                print(_json.dumps(value, indent=2, default=str)[:5000])
            else:
                print(str(value)[:5000])
        except Exception as e:
            print('FIELD ERROR', name, e)

    raw = getattr(sample, '_job', None)
    print('\n_raw type', type(raw))
    if isinstance(raw, dict):
        print(_json.dumps(raw, indent=2, default=str)[:12000])

    client = getattr(sample, '_client', None)
    print('client', type(client))
    print('client dir', [x for x in dir(client) if not x.startswith('__')][:200])

backends ['ionq_simulator', 'ionq_qpu']
jobs failed for backend <qiskit_ionq.ionq_backend.IonQSimulatorBackend object at 0x7f8c14255590> 'IonQSimulatorBackend' object has no attribute 'jobs'
jobs failed for backend <qiskit_ionq.ionq_backend.IonQQPUBackend object at 0x7f8208939310> 'IonQQPUBackend' object has no attribute 'jobs'
sample_backend None
sample_type <class 'NoneType'>
sample_dir []


In [14]:
all_df = pd.concat([ibm_df, ionq_df], ignore_index=True, sort=False)

for col in ['created_at', 'started_at', 'ended_at']:
    all_df[col] = pd.to_datetime(all_df[col], errors='coerce', utc=True)
all_df['runtime_ms'] = pd.to_numeric(all_df['runtime_ms'], errors='coerce')
all_df['shots'] = pd.to_numeric(all_df['shots'], errors='coerce')

csv_path = OUT_DIR / 'historic_runs_unified.csv'
json_path = OUT_DIR / 'historic_runs_unified.json'
summary = (
    all_df.groupby(['provider', 'backend', 'status'], dropna=False)['runtime_ms']
    .agg(['count', 'median', 'mean'])
    .reset_index()
    .sort_values(['provider', 'count'], ascending=[True, False])
)

all_df.to_csv(csv_path, index=False)
all_df.to_json(json_path, orient='records', date_format='iso')

print('Saved:', csv_path)
print('Saved:', json_path)
print('Rows:', len(all_df))
display(summary.head(30))
all_df.head()

Saved: /lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/MQBac/final_data/historic_runs_unified.csv
Saved: /lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/MQBac/final_data/historic_runs_unified.json
Rows: 80


,provider,backend,status,count,median,mean
0,ibmq,ibm_boston,Completed,34,246894.351000,249168.224118
5,ibmq,ibm_pittsburgh,Completed,18,33863.787000,51382.900222
7,ibmq,ibm_torino,Completed,15,3778.488000,36770.743000
1,ibmq,ibm_fez,Completed,4,48201.495000,68313.343000
3,ibmq,ibm_miami,Completed,4,486532.470500,474939.400750
2,ibmq,ibm_miami,Cancelled,3,2187.006238,2209.193660
4,ibmq,ibm_miami,Failed,1,4064.091405,4064.091405
6,ibmq,ibm_torino,Cancelled,1,1988.574269,1988.574269


,provider,job_id,backend,status,created_at,started_at,ended_at,runtime_ms,shots,benchmark_hint,...,measure_ops,gate_counts_json,qbacmet_framework,critical_path_length,connected_components,num_cliffords,num_non_cliffords,num_parameters,circuit_metrics_json,session_id
0,ibmq,d6gvatithhns7391g1eg,ibm_pittsburgh,Completed,2026-02-27 19:50:14.591476+00:00,2026-02-27 19:50:15.978210+00:00,2026-02-27 19:50:50.220745+00:00,34242.535,NaN,None,...,24,"{""sx"": 3146, ""rz"": 2672, ""cz"": 1089, ""x"": 121,...",qiskit,512,1,4381,2672,0,"{""framework"": ""qiskit"", ""critical_path_length""...",NaN
1,ibmq,d6gv5k1keb2s73bdvlm0,ibm_pittsburgh,Completed,2026-02-27 19:38:56.596788+00:00,2026-02-27 19:38:57.672018+00:00,2026-02-27 19:39:31.487533+00:00,33815.515,NaN,None,...,24,"{""sx"": 2200, ""rz"": 1759, ""cz"": 730, ""x"": 25, ""...",qiskit,412,1,2980,1759,0,"{""framework"": ""qiskit"", ""critical_path_length""...",NaN
2,ibmq,d6gv4m9keb2s73bdvkkg,ibm_pittsburgh,Completed,2026-02-27 19:36:58.134581+00:00,2026-02-27 19:36:59.060995+00:00,2026-02-27 19:37:33.566454+00:00,34505.459,NaN,None,...,24,"{""sx"": 2209, ""rz"": 1738, ""cz"": 744, ""x"": 32, ""...",qiskit,445,1,3010,1738,0,"{""framework"": ""qiskit"", ""critical_path_length""...",NaN
3,ibmq,d6gv1ae48nic73amc8b0,ibm_pittsburgh,Completed,2026-02-27 19:29:45.311504+00:00,2026-02-27 19:29:46.057064+00:00,2026-02-27 19:30:19.794150+00:00,33737.086,NaN,None,...,24,"{""sx"": 2145, ""rz"": 1754, ""cz"": 700, ""measure"":...",qiskit,383,1,2892,1754,0,"{""framework"": ""qiskit"", ""critical_path_length""...",NaN
4,ibmq,d6guuvn3o3rs73cabc7g,ibm_pittsburgh,Completed,2026-02-27 19:24:46.503387+00:00,2026-02-27 19:24:47.381003+00:00,2026-02-27 19:25:21.293062+00:00,33912.059,NaN,None,...,24,"{""sx"": 1354, ""rz"": 1088, ""cz"": 436, ""measure"":...",qiskit,245,1,1832,1088,0,"{""framework"": ""qiskit"", ""critical_path_length""...",NaN


In [15]:
hist_runtime = pd.DataFrame({
    'benchmark': all_df['benchmark_hint'],
    'size': all_df['n_qubits'],
    'backend': all_df['provider'],
    'sub_backend': all_df['backend'],
    'device': 'CPU',
    'run_mode': 'sync',
    'n_nodes': 1,
    'n_processes': 1,
    'runtime_ms': all_df['runtime_ms'],
    'job_id': all_df['job_id'],
    'status': all_df['status'],
    'created_at': all_df['created_at'],
    'started_at': all_df['started_at'],
    'ended_at': all_df['ended_at'],
})

hist_runtime = hist_runtime.dropna(subset=['runtime_ms']).copy()
runtime_path = OUT_DIR / 'historic_runs_for_estimate_runtime_df.csv'
hist_runtime.to_csv(runtime_path, index=False)

print('Saved:', runtime_path)
print('Rows:', len(hist_runtime))
hist_runtime.head()

Saved: /lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/MQBac/final_data/historic_runs_for_estimate_runtime_df.csv
Rows: 80


,benchmark,size,backend,sub_backend,device,run_mode,n_nodes,n_processes,runtime_ms,job_id,status,created_at,started_at,ended_at
0,None,156,ibmq,ibm_pittsburgh,CPU,sync,1,1,34242.535,d6gvatithhns7391g1eg,Completed,2026-02-27 19:50:14.591476+00:00,2026-02-27 19:50:15.978210+00:00,2026-02-27 19:50:50.220745+00:00
1,None,156,ibmq,ibm_pittsburgh,CPU,sync,1,1,33815.515,d6gv5k1keb2s73bdvlm0,Completed,2026-02-27 19:38:56.596788+00:00,2026-02-27 19:38:57.672018+00:00,2026-02-27 19:39:31.487533+00:00
2,None,156,ibmq,ibm_pittsburgh,CPU,sync,1,1,34505.459,d6gv4m9keb2s73bdvkkg,Completed,2026-02-27 19:36:58.134581+00:00,2026-02-27 19:36:59.060995+00:00,2026-02-27 19:37:33.566454+00:00
3,None,156,ibmq,ibm_pittsburgh,CPU,sync,1,1,33737.086,d6gv1ae48nic73amc8b0,Completed,2026-02-27 19:29:45.311504+00:00,2026-02-27 19:29:46.057064+00:00,2026-02-27 19:30:19.794150+00:00
4,None,156,ibmq,ibm_pittsburgh,CPU,sync,1,1,33912.059,d6guuvn3o3rs73cabc7g,Completed,2026-02-27 19:24:46.503387+00:00,2026-02-27 19:24:47.381003+00:00,2026-02-27 19:25:21.293062+00:00


In [16]:
train_df = all_df.copy()
train_df['runtime_ms'] = pd.to_numeric(train_df['runtime_ms'], errors='coerce')
train_df['shots'] = pd.to_numeric(train_df['shots'], errors='coerce')

num_cols = [
    'n_qubits', 'depth', 'n_ops', 'single_qubit_gates',
    'two_qubit_gates', 'three_qubit_gates', 'measure_ops', 'shots'
]
cat_cols = ['provider', 'backend', 'input_format']

train_df = train_df.dropna(subset=['runtime_ms']).copy()
train_df = train_df[train_df[num_cols].notna().any(axis=1)].copy()

X = train_df[num_cols + cat_cols]
y = train_df['runtime_ms'].astype(float)

pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('oh', OneHotEncoder(handle_unknown='ignore')),
    ]), cat_cols),
])

runtime_model = Pipeline([
    ('pre', pre),
    ('rf', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
])

runtime_model.fit(X, y)

backend_catalog = (
    train_df[['provider', 'backend', 'input_format']]
    .drop_duplicates()
    .sort_values(['provider', 'backend'])
    .reset_index(drop=True)
)


def circuit_features_from_qasm(qasm_path: str, shots: int = 1024):
    qc = QuantumCircuit.from_qasm_str(Path(qasm_path).read_text())
    counts = qc.count_ops()
    one_q = 0
    two_q = 0
    three_q = 0
    for inst, qargs, _ in qc.data:
        if len(qargs) == 1:
            one_q += 1
        elif len(qargs) == 2:
            two_q += 1
        elif len(qargs) >= 3:
            three_q += 1
    return {
        'n_qubits': int(qc.num_qubits),
        'depth': int(qc.depth()),
        'n_ops': int(qc.size()),
        'single_qubit_gates': one_q,
        'two_qubit_gates': two_q,
        'three_qubit_gates': three_q,
        'measure_ops': int(counts.get('measure', 0)),
        'shots': int(shots),
    }


def estimate_runtime_all_backends(qasm_path: str, shots: int = 1024) -> pd.DataFrame:
    feats = circuit_features_from_qasm(qasm_path, shots=shots)
    rows = []
    for _, row in backend_catalog.iterrows():
        rows.append({
            **feats,
            'provider': row['provider'],
            'backend': row['backend'],
            'input_format': row['input_format'] if pd.notna(row['input_format']) else 'qasm',
        })
    X_new = pd.DataFrame(rows)
    pred = runtime_model.predict(X_new)
    out = X_new[['provider', 'backend']].copy()
    out['pred_runtime_ms'] = pred
    return out.sort_values('pred_runtime_ms').reset_index(drop=True)

print('Training rows:', len(train_df))
print('Backends modeled:', len(backend_catalog))
print("Ready: estimate_runtime_all_backends('/lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/each_circ.qasm')")

/lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/MQBac/mqbacVirtEnv/lib64/python3.11/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['shots']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training rows: 80
Backends modeled: 6
Ready: estimate_runtime_all_backends('/lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/each_circ.qasm')
